
# PDF QA Sistemi

## Amaç
Bu projede PDF dosyaları üzerinden soru-cevap sistemi geliştirilmiştir.

Sistem:
- PDF dosyasını okur
- Metni işler
- Kullanıcının sorduğu soruya PDF içeriğine göre cevap verir

## Kullanılan Teknolojiler
- Python
- LangChain
- FAISS
- Sentence Transformers
- Transformers



# 1. Gerekli Kütüphanelerin Kurulması


In [ ]:

!pip install PyPDF2
!pip install langchain
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install torch



# 2. PDF Dosyasının Yüklenmesi

Bu aşamada PDF dosyasının içeriği okunur.


In [ ]:

from PyPDF2 import PdfReader

pdf_path = "ornek.pdf"   # Buraya kendi PDF dosyanızın adını yazın

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print(text[:2000])



# 3. Metnin Parçalara Ayrılması

Uzun metinler daha iyi işlenebilmesi için küçük parçalara bölünür.


In [ ]:

from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print("Toplam Parça Sayısı:", len(chunks))
print(chunks[0])



# 4. Embedding Oluşturma

Metin parçaları sayısal vektörlere dönüştürülür.


In [ ]:

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(chunks)

print("Embedding Boyutu:", embeddings.shape)



# 5. FAISS Veritabanı Oluşturma

Benzer metinleri hızlı bulabilmek için FAISS kullanılır.


In [ ]:

import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS veritabanına eklenen parça sayısı:", index.ntotal)



# 6. Soru-Cevap Sistemi

Kullanıcının sorusuna en uygun PDF parçası bulunur ve cevap üretilir.


In [ ]:

from transformers import pipeline

qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)


In [ ]:

question = input("Sorunuzu girin: ")

question_embedding = embedding_model.encode([question])

k = 1

distances, indices = index.search(
    np.array(question_embedding),
    k
)

relevant_text = chunks[indices[0][0]]

prompt = f'''
Aşağıdaki metne göre soruyu cevapla.

Metin:
{relevant_text}

Soru:
{question}
'''

result = qa_pipeline(
    prompt,
    max_length=100
)

print("\nCevap:")
print(result[0]['generated_text'])



# 7. Sonuç

Bu projede PDF dosyaları üzerinde çalışan bir soru-cevap sistemi geliştirilmiştir.

Sistem:
- PDF içeriğini okuyabilmektedir
- Metni işleyebilmektedir
- Kullanıcı sorularına uygun cevap verebilmektedir

Bu yöntem özellikle:
- Doküman analizi
- Akademik araştırmalar
- Rapor inceleme
- Bilgi erişim sistemleri

gibi alanlarda kullanılabilir.
